# 01 · Data, splits, corpus and metrics (Phase 1, owner: M1)

Runs Prompts 1-4 on Kaggle. Before running:
1. **Settings → Internet: On** (needs a phone-verified Kaggle account). Without internet, add Kaggle
   Datasets that contain `train.jsonl`, `shared_task_dev.jsonl`, `wiki-pages.zip` (or the unzipped
   `wiki-NNN.jsonl` files) and `hotpot_dev_distractor_v1.json`; the scripts find them under `/kaggle/input`.
2. Get the code (private repo `group-mauve/claim-verification-pipeline`): **Add-ons → Secrets → Add Secret**,
   label `GITHUB_TOKEN`, value = a GitHub fine-grained token with read-only *Contents* access to the repo,
   and tick it for this notebook. Fallback: upload codebase/ as a private Dataset `fever-project-code`.
3. Accelerator: **None** is enough for this notebook (no GPU needed; save your GPU quota).

Expected run time: about 1-1.5 hours (most of it parsing 5.4M wiki pages and the decoy BM25 search).

In [ ]:
import os, shutil, subprocess
REPO_URL = "https://github.com/group-mauve/claim-verification-pipeline.git"
BRANCH = "main"
CODE_DATASET = "/kaggle/input/fever-project-code"   # fallback: code uploaded as a Kaggle Dataset
WORK = "/kaggle/working/project"

def git(*args, token=None):
    """Run git; if a token is given, use it for this call only and hide it from any error."""
    cmd = ["git"] + [a.replace("https://", f"https://x-access-token:{token}@") if token and a == REPO_URL else a for a in args]
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token, "***") if token else r.stdout + r.stderr
    print(out.strip())
    if r.returncode != 0:
        raise RuntimeError("git failed (see message above). Check the GITHUB_TOKEN secret and that Internet is on.")

token = None
try:  # private repo: token stored in Add-ons -> Secrets as GITHUB_TOKEN
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret attached; trying the code dataset instead.")

if token:
    if not os.path.exists(WORK):
        git("clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORK, token=token)
        git("-C", WORK, "remote", "set-url", "origin", REPO_URL)   # token is never stored in .git/config
    else:
        git("-C", WORK, "pull", REPO_URL, BRANCH, token=token)    # pick up teammates' latest pushes
elif not os.path.exists(WORK):
    if not os.path.exists(CODE_DATASET):
        raise FileNotFoundError("Attach the GITHUB_TOKEN secret, or add the code dataset at " + CODE_DATASET)
    shutil.copytree(CODE_DATASET, WORK)
os.chdir(WORK)
print(os.getcwd(), sorted(os.listdir()))

In [ ]:
# The official FEVER scorer's setup.py no longer builds with current pip, so instead of
# `pip install` we clone it and put its package folder (the one holding fever/scorer.py) on the path.
import os, glob, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "pytest"], check=True)
SCORER_DIR = "/kaggle/working/fever-scorer"
if not os.path.exists(SCORER_DIR):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/sheffieldnlp/fever-scorer", SCORER_DIR], check=True)
hits = glob.glob(f"{SCORER_DIR}/**/fever/scorer.py", recursive=True)
assert hits, "fever/scorer.py not found in the cloned repo"
pkg_root = os.path.dirname(os.path.dirname(hits[0]))
os.environ["PYTHONPATH"] = pkg_root + os.pathsep + os.environ.get("PYTHONPATH", "")  # seen by !python / !pytest
sys.path.insert(0, pkg_root)
from fever.scorer import fever_score
print("official scorer importable from", pkg_root)

## Prompt 4 · metric unit tests (slide examples + official FEVER scorer)
The official-scorer test needs `shared_task_dev.jsonl`, so it is skipped on the first run and checked again after the download.

In [ ]:
!python -m pytest -q -rs

## Prompt 1 · download, parse Wikipedia to Parquet, verify counts

In [ ]:
!python -m src.data.download --config configs/data.yaml

## Prompt 2 · train / validation / test splits

In [ ]:
!python -m src.data.splits --config configs/data.yaml

## Prompt 3 · restricted corpus (~50k pages, <= 1M sentences)

In [ ]:
!python -m src.data.build_corpus --config configs/data.yaml

In [ ]:
import json, pandas as pd
stats = json.load(open("data/corpus/corpus_stats.json"))
print(json.dumps(stats, indent=2))
sents = pd.read_parquet("data/corpus/sentences.parquet")
pages = pd.read_parquet("data/corpus/pages.parquet")
print(sents.shape, pages.shape)
print(pages["source"].value_counts())
sents.sample(5, random_state=42)

## Prompt 4 again · official scorer comparison now that the dev file exists

In [ ]:
!python -m pytest -q -rs tests/test_metrics.py

## Package outputs for Kaggle Datasets

`fever_corpus_v1/` (about 0.5-1 GB) is what every later notebook needs: the corpus, the splits and the
small raw files. The 5.4M-page wiki Parquet (several GB) is only needed to rebuild the corpus; keep it
as a second dataset only if you have space.

After this cell: **Save Version → Save & Run All (Commit)**. When it finishes, open the notebook's
**Output** tab → **New Dataset**, name it `fever-corpus-v1`, and keep it private. Later notebooks add it
as an input at `/kaggle/input/fever-corpus-v1`.

In [ ]:
import shutil, glob, os
PKG = "/kaggle/working/fever_corpus_v1"
os.makedirs(PKG, exist_ok=True)
shutil.copytree("data/corpus", f"{PKG}/corpus", dirs_exist_ok=True, ignore=shutil.ignore_patterns("_decoy_cache.json"))
shutil.copytree("data/splits", f"{PKG}/splits", dirs_exist_ok=True)
os.makedirs(f"{PKG}/raw", exist_ok=True)
for f in ["train.jsonl", "shared_task_dev.jsonl", "hotpot_dev_distractor_v1.json"]:
    shutil.copy2(f"data/raw/{f}", f"{PKG}/raw/{f}")
shutil.copy2("data/README.md", f"{PKG}/README.md")

KEEP_WIKI_PARQUET = False   # True = also keep data/wiki_sentences in the notebook output
# Delete the large files from the output so the version saves quickly.
if os.path.exists("data/raw/wiki-pages.zip"):
    os.remove("data/raw/wiki-pages.zip")
shutil.rmtree("/kaggle/working/fever-scorer", ignore_errors=True)
if not KEEP_WIKI_PARQUET:
    shutil.rmtree("data/wiki_sentences", ignore_errors=True)
!du -sh /kaggle/working/* /kaggle/working/fever_corpus_v1/*